# ALMA Interferometric ASDM guide

## Preparation

### Import Xarray

We will first use the standard API of Xarray Data Trees to open an example ASDM. Importing Xradio at this point is not needed, but it must be installed together with the optional dependencies of its ASDM backend (in particular [pyasdm](https://github.com/casangi/pyasdm)), for example with `pip install "xradio[asdm]"` (or `pip install "xradio[all]"`).

The next cell installs `xradio[all]` only if `xradio` or `pyasdm` cannot be imported (for example on Google Colab). An existing installation is not upgraded or replaced.

In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    import pyasdm  # needed by the xradio_asdm engine (optional dependency)

    import xradio
except ImportError as exc:
    # For example on Google Colab. Install only what is missing: an existing xradio
    # installation (such as a development install) is not upgraded or replaced.
    print(f"{exc}. Installing xradio with all its optional dependencies.")
    subprocess.run([sys.executable, "-m", "pip", "install", "xradio[all]"], check=True)

print("Using xradio version", importlib.metadata.version("xradio"))

Xradio adds an `xradio_asdm` custom backend to Xarray. We can see it in the list of backends available in Xarray together with for example `zarr`. When Xradio is installed with the dependencies of the ASDM backend (`pip install "xradio[asdm]"`), `xradio_asdm` can be given to the `engine` parameter of `xarray.open_datatree()` to open ASDMs.

In [ ]:
import xarray as xr

xr.backends.list_engines()

### Data Download

The example dataset is a small ALMA interferometric ASDM (execution block `uid___A002_X10f6291_X12d9`, about 3.4 MB zipped). Like the datasets of the other guides, it is downloaded with `toolviper.utils.data.download()` into the current directory, unless it is already there. The cell below first checks that the dataset is listed in the toolviper download manifest (`toolviper.utils.data.get_files()`), and only then requests it.

**Temporary fallback:** while this dataset is not yet available in the toolviper download manifest, the cell below falls back to a copy hosted on Google Drive. That fallback needs the third-party package [gdown](https://github.com/wkentaro/gdown), which is not a dependency of Xradio and is **not** installed automatically. If the dataset cannot be downloaded with toolviper and `gdown` is missing, the cell stops with an error: install `gdown` yourself (`pip install gdown`) and run the cell again. The fallback will be removed once the dataset is in the toolviper manifest.

In [ ]:
import pathlib
import zipfile

import toolviper

asdm_name = "uid___A002_X10f6291_X12d9"

# Google Drive id of a zipped copy of the example ASDM. Temporary fallback, only used
# while the dataset is not available in the toolviper download manifest.
gdrive_fallback_file_id = "1sup-z1anlh_xYpTlhNalVhtOvbxzAg3S"


def asdm_is_present(path: str) -> bool:
    return pathlib.Path(path, "ASDM.xml").is_file()


if not asdm_is_present(asdm_name):
    if asdm_name in toolviper.utils.data.get_files():
        try:
            toolviper.utils.data.download(file=asdm_name)
        except RuntimeError as exc:
            print(f"Could not download {asdm_name} with toolviper: {exc}")
    else:
        print(f"{asdm_name} is not available in the toolviper download manifest.")

if not asdm_is_present(asdm_name):
    try:
        import gdown
    except ImportError as exc:
        raise RuntimeError(
            f"The example ASDM {asdm_name} could not be downloaded with toolviper, and "
            "the temporary Google Drive fallback needs the package 'gdown', which is not "
            "installed. Install it with `pip install gdown` and run this cell again."
        ) from exc

    print(f"Downloading {asdm_name} from its temporary Google Drive copy.")
    try:
        zip_path = gdown.download(
            id=gdrive_fallback_file_id, output=f"{asdm_name}.zip", quiet=True
        )
    except Exception as exc:  # for example gdown.exceptions.FileURLRetrievalError
        raise RuntimeError(
            f"Failed to download {asdm_name} from Google Drive: {exc}"
        ) from exc
    if zip_path is None:
        raise RuntimeError(f"Failed to download {asdm_name} from Google Drive.")
    with zipfile.ZipFile(zip_path) as zref:
        zref.extractall()
    pathlib.Path(zip_path).unlink()

if not asdm_is_present(asdm_name):
    raise RuntimeError(f"The example ASDM {asdm_name} is not available.")
print(f"Using the example ASDM {asdm_name}")

## Open an ASDM as a Processing Set

We can open the ASDM with `open_datatree()`. One relevant parameter is `include_processor_types` which selects the type of data (processor type) to include in the output DataTree. In this example we take only correlator data, not including the other types: `"RADIOMETER"` and `"SPECTROMETER"`.

The ASDM is opened lazily: `open_datatree()` reads the ASDM metadata tables to build the MSv4s, but the correlated data (`VISIBILITY`) and `FLAG` are read from the ASDM binary data only when they are accessed or loaded, and `UVW` is computed from the metadata (antenna positions, phase center and times) when it is accessed. `WEIGHT` is currently a constant 1.0, as the ASDM binary data have no weights. As with other Xarray backends, a `chunks` parameter can be given to `open_datatree()` to get Dask arrays that are read chunk by chunk (see the section on saving the Processing Set below).

In [ ]:
asdm_ps_xdt = xr.open_datatree(
    "uid___A002_X10f6291_X12d9",
    engine="xradio_asdm",
    include_processor_types=["CORRELATOR"],
)

## Explore the Processing Set

The processing set opened from the ASDM can be explored via the standard Xarray DataTree interface and also via the `xr_ps` accessor:

In [ ]:
asdm_ps_xdt.xr_ps.summary()

In [ ]:
asdm_ps_xdt

### Measurement Sets

The Processing Set Data Tree contains Data trees of MSv4s, including the correlated dataset and its sub-datasets. The MSv4 nodes are named `{asdm_name}_{index}`, where the index is zero-padded to a common width (for example `_0` to `_7` for 8 MSv4s, but `_00` to `_11` for 12 MSv4s). Here we take the first MSv4:

In [ ]:
msv4_name = next(iter(asdm_ps_xdt.children))  # "uid___A002_X10f6291_X12d9_0"
msv4_xdt_0 = asdm_ps_xdt[msv4_name]
msv4_xdt_0.observation_info

#### Correlated dataset:

In [ ]:
corr_xds = msv4_xdt_0.ds
corr_xds

#### Sub-datasets

In [ ]:
antenna_xds = msv4_xdt_0["antenna_xds"]
antenna_xds

In [ ]:
field_and_source_xds = msv4_xdt_0.xr_ms.get_field_and_source_xds()
field_and_source_xds

In [ ]:
field_and_source_xds.FIELD_PHASE_CENTER_DIRECTION

In [ ]:
asdm_ps_xdt.xr_ps.plot_antenna_positions_2d()

## Opening ASDMs with filters by processor type and spectral resolution type.


In the following example we open a more complete Processing Set including the MSv4s that correspond to correlator type `RADIOMETER` (in addition to the `CORRELATOR` type used above). We also include all the spectral resolution types.

In [ ]:
full_asdm_ps_xdt = xr.open_datatree(
    "uid___A002_X10f6291_X12d9",
    include_processor_types=["CORRELATOR", "RADIOMETER"],
    include_spectral_resolution_types=[
        "FULL_RESOLUTION",
        "CHANNEL_AVERAGE",
        "BASEBAND_WIDE",
    ],
    engine="xradio_asdm",
)

In [ ]:
full_asdm_ps_xdt.xr_ps.summary()

## Other options

Other options supported include `partition_scheme` (optional partitioning axes, among `"fieldId"` (the default), `"scanNumber"` and `"subscanNumber"`), `with_pointing` and `pointing_for_only_spectral_resolution_types`. Support for pointing is currently under development.

The function that implements `open_datatree()` for ASDMs is also available as `xradio.measurement_set.open_asdm()`. Its [API documentation](../api.rst) lists all the parameters accepted by the `xradio_asdm` engine. There is however no need to use this function directly: one can simply rely on Xarray's `open_datatree()`, as seen above.

## Saving the Processing Set to other formats

Once the ASDM is opened as a Processing Set, it can be saved to other formats supported in Xarray, for example Zarr.

Writing the Processing Set reads all its data from the ASDM. If the DataTree was opened without `chunks` (as `asdm_ps_xdt` above), each data variable of each MSv4 is read whole into memory before being written. That is fine for this small example, but for large ASDMs it can require tens or hundreds of GB of memory. When the ASDM is opened with `chunks`, the data variables are Dask arrays and the data are read from the ASDM and written chunk by chunk, so the memory needed is bounded by the chunk size. With `chunks={}`, every MSv4 gets time chunks of one size (the last chunk can be smaller), shared by all its data variables: about the number of integrations of its largest binary data file (BDF), fewer when needed to fit the Dask chunk size. The chunks line up with the BDFs only when all the BDFs of the MSv4 have the same number of integrations (the last one can have fewer). Otherwise some chunks span two BDFs, which makes reading them slower. With `chunks="auto"`, as below, Dask can also merge several of these chunks into one, up to its chunk size, which gives fewer and larger chunks (and Zarr chunks) when the BDFs are small, as in this example. Explicit chunk sizes, for example `chunks={"time": 10}`, can also be given. Chunks that split BDFs make reading slower, and xarray warns when explicit chunks "separate the stored chunks", that is the chunks that `chunks={}` would give. `mode="w"` overwrites the output if it already exists (for example when this notebook is run again).

In [ ]:
asdm_ps_chunked_xdt = xr.open_datatree(
    "uid___A002_X10f6291_X12d9",
    engine="xradio_asdm",
    include_processor_types=["CORRELATOR"],
    chunks="auto",
)
asdm_ps_chunked_xdt.to_zarr(
    "asdm_uid___A002_X10f6291_X12d9_saved_as_zarr.zarr", mode="w"
)